# Air Quality Data in India — Exploratory Data Analysis

**Reference dataset:** [Kaggle — Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)

This notebook performs a structured exploratory analysis of city-level air-quality measurements.

> **Data integrity note:** The notebook uses the official `city_day.csv` if it is present in the same folder. Otherwise, it uses `sample_air_quality_data.csv` only to demonstrate that the code runs. The included sample is synthetic and must not be used for real-world conclusions.

## 1. Import libraries and configure the notebook

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Output folder:", os.path.abspath(OUTPUT_DIR))


## 2. Load the dataset

In [ ]:
OFFICIAL_FILE = "city_day.csv"
SAMPLE_FILE = "sample_air_quality_data.csv"

if os.path.exists(OFFICIAL_FILE):
    data_path = OFFICIAL_FILE
    data_source_label = "OFFICIAL KAGGLE DATASET"
elif os.path.exists(SAMPLE_FILE):
    data_path = SAMPLE_FILE
    data_source_label = "ILLUSTRATIVE SAMPLE — NOT REAL-WORLD DATA"
    warnings.warn(
        "city_day.csv was not found. The notebook is using synthetic sample data. "
        "Download the official Kaggle file before preparing final conclusions.",
        UserWarning
    )
else:
    raise FileNotFoundError(
        "Could not find city_day.csv or sample_air_quality_data.csv. "
        "Put the official Kaggle city_day.csv beside this notebook."
    )

df_raw = pd.read_csv(data_path)
print("Data source:", data_source_label)
print("File:", data_path)
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]:,}")
display(df_raw.head(10))


## 3. Understand the structure

In [ ]:
print("Column names:")
print(df_raw.columns.tolist())
print("\nData types and non-null counts:")
df_raw.info()
print("\nRandom sample of records:")
display(df_raw.sample(min(8, len(df_raw)), random_state=42))


## 4. Check expected columns and standardize names

In [ ]:
# Trim accidental leading/trailing spaces in column names.
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

expected_columns = [
    "City", "Date", "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
    "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene", "AQI", "AQI_Bucket"
]
missing_expected = [c for c in expected_columns if c not in df.columns]
extra_columns = [c for c in df.columns if c not in expected_columns]

print("Expected columns missing from this file:", missing_expected or "None")
print("Additional columns:", extra_columns or "None")


## 5. Missing-value analysis

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)
missing_summary.to_csv(os.path.join(OUTPUT_DIR, "missing_value_summary.csv"))

missing_plot = missing_summary[missing_summary["missing_count"] > 0]
if not missing_plot.empty:
    plt.figure(figsize=(11, 5))
    missing_plot["missing_percent"].sort_values().plot(kind="barh")
    plt.title("Percentage of Missing Values by Column")
    plt.xlabel("Missing values (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "missing_values.png"), dpi=150)
    plt.show()
else:
    print("No missing values were found.")


## 6. Duplicate and date-quality checks

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)

if "City" in df.columns and "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    invalid_dates = int(df["Date"].isna().sum())
    print("Invalid or missing dates:", invalid_dates)
    duplicate_city_date = int(df.duplicated(subset=["City", "Date"]).sum())
    print("Repeated City + Date pairs:", duplicate_city_date)
else:
    print("City and/or Date column missing; city-date checks skipped.")

numeric_candidates = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
                      "Benzene", "Toluene", "Xylene", "AQI"]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


## 7. Descriptive statistics

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
if numeric_columns:
    descriptive_stats = df[numeric_columns].describe().T
    descriptive_stats["missing_count"] = df[numeric_columns].isna().sum()
    display(descriptive_stats.round(3))
    descriptive_stats.to_csv(os.path.join(OUTPUT_DIR, "descriptive_statistics.csv"))
else:
    print("No numeric columns were found.")


## 8. City and date coverage

In [ ]:
if "City" in df.columns:
    print("Unique cities:", df["City"].nunique(dropna=True))
    display(df["City"].value_counts(dropna=False).head(30).to_frame("record_count"))

if "Date" in df.columns and df["Date"].notna().any():
    print("First recorded date:", df["Date"].min())
    print("Last recorded date:", df["Date"].max())
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Month_Name"] = df["Date"].dt.month_name()
    df["Day_of_Week"] = df["Date"].dt.day_name()
    df["YearMonth"] = df["Date"].dt.to_period("M").astype(str)


## 9. AQI distribution

In [ ]:
if "AQI" in df.columns:
    aqi_values = df["AQI"].dropna()
    if not aqi_values.empty:
        print(aqi_values.describe().round(2))
        plt.figure(figsize=(10, 5))
        sns.histplot(aqi_values, bins=35, kde=True, color="steelblue")
        plt.title("Distribution of Air Quality Index (AQI)")
        plt.xlabel("AQI")
        plt.ylabel("Number of records")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "aqi_distribution.png"), dpi=150)
        plt.show()

        plt.figure(figsize=(8, 4))
        sns.boxplot(x=aqi_values, color="steelblue")
        plt.title("AQI Box Plot")
        plt.xlabel("AQI")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "aqi_boxplot.png"), dpi=150)
        plt.show()
    else:
        print("AQI column exists but has no numeric values.")
else:
    print("AQI column is not available.")


## 10. AQI category distribution

In [ ]:
bucket_col = "AQI_Bucket" if "AQI_Bucket" in df.columns else None
if bucket_col:
    bucket_counts = df[bucket_col].fillna("Missing").value_counts()
    display(bucket_counts.to_frame("record_count"))
    bucket_counts.to_csv(os.path.join(OUTPUT_DIR, "aqi_category_counts.csv"))

    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x=bucket_col, order=df[bucket_col].value_counts().index, color="steelblue")
    plt.title("Records by AQI Category")
    plt.xlabel("AQI category")
    plt.ylabel("Record count")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "aqi_category_distribution.png"), dpi=150)
    plt.show()
else:
    print("AQI_Bucket column is not available.")


## 11. Average AQI by city

In [ ]:
if {"City", "AQI"}.issubset(df.columns):
    city_aqi = (
        df.groupby("City", dropna=True)["AQI"]
          .agg(["count", "mean", "median", "min", "max"])
          .sort_values("mean", ascending=False)
    )
    city_aqi.columns = ["record_count", "average_aqi", "median_aqi", "minimum_aqi", "maximum_aqi"]
    display(city_aqi.round(2))
    city_aqi.to_csv(os.path.join(OUTPUT_DIR, "city_aqi_summary.csv"))

    top_cities = city_aqi.head(10).sort_values("average_aqi")
    if not top_cities.empty:
        plt.figure(figsize=(10, 6))
        plt.barh(top_cities.index.astype(str), top_cities["average_aqi"])
        plt.title("10 Cities with Highest Average AQI")
        plt.xlabel("Average AQI")
        plt.ylabel("City")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "top_cities_average_aqi.png"), dpi=150)
        plt.show()

    bottom_cities = city_aqi.tail(10).sort_values("average_aqi", ascending=False)
    if not bottom_cities.empty:
        plt.figure(figsize=(10, 6))
        plt.barh(bottom_cities.index.astype(str), bottom_cities["average_aqi"])
        plt.title("10 Cities with Lowest Average AQI")
        plt.xlabel("Average AQI")
        plt.ylabel("City")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "bottom_cities_average_aqi.png"), dpi=150)
        plt.show()
else:
    print("City and/or AQI columns are not available.")


## 12. Yearly AQI trend

In [ ]:
if {"Year", "AQI"}.issubset(df.columns):
    yearly_aqi = df.groupby("Year")["AQI"].agg(["count", "mean", "median"]).dropna()
    display(yearly_aqi.round(2))
    yearly_aqi.to_csv(os.path.join(OUTPUT_DIR, "yearly_aqi_summary.csv"))

    if not yearly_aqi.empty:
        plt.figure(figsize=(9, 5))
        plt.plot(yearly_aqi.index, yearly_aqi["mean"], marker="o")
        plt.title("Average AQI by Year")
        plt.xlabel("Year")
        plt.ylabel("Average AQI")
        plt.xticks(yearly_aqi.index)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "yearly_aqi_trend.png"), dpi=150)
        plt.show()
else:
    print("Date or AQI information is unavailable.")


## 13. Monthly and seasonal patterns

In [ ]:
if {"Month", "AQI"}.issubset(df.columns):
    monthly_aqi = df.groupby("Month")["AQI"].mean().reindex(range(1, 13))
    display(monthly_aqi.to_frame("average_aqi").round(2))
    plt.figure(figsize=(10, 5))
    plt.plot(monthly_aqi.index, monthly_aqi.values, marker="o")
    plt.title("Average AQI by Calendar Month")
    plt.xlabel("Month number")
    plt.ylabel("Average AQI")
    plt.xticks(range(1, 13))
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "monthly_aqi_pattern.png"), dpi=150)
    plt.show()
else:
    print("Month and/or AQI information is unavailable.")


## 14. Weekday pattern

In [ ]:
if {"Day_of_Week", "AQI"}.issubset(df.columns):
    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    weekday_aqi = df.groupby("Day_of_Week")["AQI"].mean().reindex(weekday_order).dropna()
    display(weekday_aqi.to_frame("average_aqi").round(2))
    if not weekday_aqi.empty:
        plt.figure(figsize=(9, 4))
        weekday_aqi.plot(kind="bar")
        plt.title("Average AQI by Day of Week")
        plt.xlabel("Day of week")
        plt.ylabel("Average AQI")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "weekday_aqi_pattern.png"), dpi=150)
        plt.show()
else:
    print("Date or AQI information is unavailable.")


## 15. Pollutant distributions

In [ ]:
pollutant_columns = [c for c in numeric_candidates if c in df.columns and c != "AQI"]
available_pollutants = [c for c in pollutant_columns if df[c].notna().any()]

if available_pollutants:
    for start in range(0, len(available_pollutants), 4):
        subset = available_pollutants[start:start + 4]
        fig, axes = plt.subplots(len(subset), 1, figsize=(9, 3 * len(subset)))
        if len(subset) == 1:
            axes = [axes]
        for ax, col in zip(axes, subset):
            sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax, color="steelblue")
            ax.set_title(f"Distribution of {col}")
            ax.set_xlabel(col)
        plt.tight_layout()
        plt.show()
else:
    print("No pollutant columns with numeric data were found.")


## 16. Pollutant correlation analysis

In [ ]:
correlation_columns = [c for c in pollutant_columns + (["AQI"] if "AQI" in df.columns else [])
                       if c in df.columns and df[c].notna().sum() > 1]
if len(correlation_columns) >= 2:
    corr = df[correlation_columns].corr(numeric_only=True)
    display(corr.round(2))
    plt.figure(figsize=(12, 9))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", square=False)
    plt.title("Correlation Matrix: AQI and Pollutant Measurements")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "pollutant_correlation_heatmap.png"), dpi=150)
    plt.show()
    corr.to_csv(os.path.join(OUTPUT_DIR, "pollutant_correlation_matrix.csv"))
else:
    print("Not enough numeric columns for correlation analysis.")


## 17. Pollutant levels across cities

In [ ]:
if "City" in df.columns and available_pollutants:
    selected = [c for c in ["PM2.5", "PM10", "NO2", "CO", "SO2"] if c in available_pollutants]
    if selected:
        city_pollutants = df.groupby("City")[selected].mean()
        display(city_pollutants.round(2))
        plt.figure(figsize=(12, max(5, len(city_pollutants) * 0.28)))
        sns.heatmap(city_pollutants, cmap="YlOrRd", annot=False)
        plt.title("Average Pollutant Measurements by City")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "city_pollutant_heatmap.png"), dpi=150)
        plt.show()
else:
    print("City or pollutant measurements are unavailable.")


## 18. Compare 2019 and 2020

In [ ]:
if {"Year", "AQI"}.issubset(df.columns):
    year_comparison = df[df["Year"].isin([2019, 2020])].groupby("Year")["AQI"].agg(["count", "mean", "median"])
    if {2019, 2020}.issubset(set(year_comparison.index)):
        display(year_comparison.round(2))
        year_comparison.to_csv(os.path.join(OUTPUT_DIR, "aqi_2019_2020_comparison.csv"))
        year_comparison["mean"].plot(kind="bar", figsize=(6, 4))
        plt.title("Average AQI: 2019 vs 2020")
        plt.xlabel("Year")
        plt.ylabel("Average AQI")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "aqi_2019_2020_comparison.png"), dpi=150)
        plt.show()
    else:
        print("Both 2019 and 2020 are not available; comparison skipped.")
else:
    print("Year and/or AQI information is unavailable.")


## 19. Check potential outliers using the IQR rule

In [ ]:
outlier_rows = []
for col in ["PM2.5", "PM10", "NO2", "CO", "SO2", "AQI"]:
    if col in df.columns:
        values = df[col].dropna()
        if len(values) >= 4:
            q1, q3 = values.quantile([0.25, 0.75])
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = int(((values < lower) | (values > upper)).sum())
            outlier_rows.append({
                "column": col, "q1": q1, "q3": q3, "iqr": iqr,
                "lower_bound": lower, "upper_bound": upper, "potential_outlier_count": count
            })

outlier_summary = pd.DataFrame(outlier_rows)
if not outlier_summary.empty:
    display(outlier_summary.round(3))
    outlier_summary.to_csv(os.path.join(OUTPUT_DIR, "potential_outlier_summary.csv"), index=False)
else:
    print("Not enough values for the IQR outlier check.")

print("Note: An outlier is not automatically an error. Investigate it in context before removing it.")


## 20. Create a cleaned export without silently imputing values

In [ ]:
# This export standardizes the date and numeric types and removes exact duplicate rows.
# It deliberately leaves missing values intact so the treatment can be explained in the report.
cleaned = df.copy()
cleaned = cleaned.drop_duplicates()

if "Date" in cleaned.columns:
    cleaned["Date"] = pd.to_datetime(cleaned["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

cleaned_path = os.path.join(OUTPUT_DIR, "air_quality_cleaned.csv")
cleaned.to_csv(cleaned_path, index=False)
print("Saved cleaned export to:", cleaned_path)
print("Rows before:", len(df), "| Rows after exact-duplicate removal:", len(cleaned))


## 21. Generate a concise findings checklist

In [ ]:
findings = []
findings.append(f"Loaded source label: {data_source_label}")
findings.append(f"Dataset size: {df.shape[0]:,} rows and {df.shape[1]:,} columns.")
if "City" in df.columns:
    findings.append(f"Number of cities represented: {df['City'].nunique(dropna=True)}.")
if "Date" in df.columns and df["Date"].notna().any():
    findings.append(f"Date coverage: {df['Date'].min().date()} to {df['Date'].max().date()}.")
if "AQI" in df.columns and df["AQI"].notna().any():
    findings.append(f"Mean AQI across available records: {df['AQI'].mean():.2f}.")
    if "City" in df.columns:
        mean_by_city = df.groupby("City")["AQI"].mean().dropna()
        if not mean_by_city.empty:
            findings.append(f"Highest city mean AQI in this file: {mean_by_city.idxmax()} ({mean_by_city.max():.2f}).")
            findings.append(f"Lowest city mean AQI in this file: {mean_by_city.idxmin()} ({mean_by_city.min():.2f}).")

findings_text = "\n".join(f"- {item}" for item in findings)
print(findings_text)
with open(os.path.join(OUTPUT_DIR, "findings_checklist.txt"), "w", encoding="utf-8") as f:
    f.write("AUTOMATIC SUMMARY — verify against official data before reporting\n\n")
    f.write(findings_text + "\n")


## 22. Conclusion — write after using the official dataset

After running the notebook with the official Kaggle `city_day.csv`, summarize:
1. Which cities have the highest and lowest average AQI in this dataset.
2. Whether average AQI changes across years or months.
3. Which AQI categories are most common.
4. Which pollutant variables are correlated with AQI.
5. The limitations caused by missing values, uneven city coverage, and observational data.

Do not copy the automatic summary as a final scientific conclusion without checking the tables and charts. Correlation does not prove causation, and results from the included synthetic sample are for testing only.